In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pylab as plt

def gaussian_pdf(x, mu, sigma):
    term1 = 1 / (sigma * np.sqrt(2 * np.pi))
    term2 = np.exp(-0.5 * ((x - mu) / sigma)**2)
    return term1 * term2

In [ ]:
param_sets = ((-1, 1, 1), (-.5, .5, 1), (-1, 1, 2))
x = np.linspace(-5, 5, 500)

for i, params in enumerate(param_sets):
    plt.subplot(1, 3, i+1)

    m1, m2, sd = params
    y1 = gaussian_pdf(x, m1, sd)
    y2 = gaussian_pdf(x, m2, sd)
    dprime = (m2 - m1) / sd
    C = -.5 * (m2 + m1)
    
    plt.plot(x, y1, label='Decrease', ls='--', c='k')
    plt.plot(x, y2, label='Increase', c='k')
    plt.title('$d^\\prime$=%.02f; C=%.02f' % (dprime, C))
    plt.xlabel('Perceived Pitch of Probe (JNDs from Standard)')
    plt.ylabel('Probability Density')
    plt.grid(True)
    plt.legend(title='Pitch Change')
    plt.xlim(-5, 5)
    plt.ylim(-.005, .5)

plt.gcf().set_size_inches(20, 4)

In [ ]:
color_spec = '#00CED1' #Dark turquoise
color_temp = '#D11141' #Dark pink

# Construct means and standard deviations of spectral cue to pitch change representing different difficulties
titles = ('Pitch Shifted by JND', 'Pitch Shifted by Half JND', 'Pitch Shifted by JND with Added Noise')
param_sets = ((-1, 1, 1), (-.5, .5, 1), (-1, 1, 2))

# Suppose a late temporal cue has a mean of -5 JNDs and a standard deviation of 8
mt = -5
st = 6

x = np.linspace(-5, 5, 1000)
fig, axes = plt.subplots(1, 3, figsize=(18.5, 5))
for i, params in enumerate(param_sets):
    
    # Mix pitch cue with timing cue
    m1, m2, sd = params
    mc1 = (m1/sd**2 + mt/st**2) / (1/sd**2 + 1/st**2)
    mc2 = (m2/sd**2 + mt/st**2) / (1/sd**2 + 1/st**2)
    sc = np.sqrt(1 / ((1/sd**2) + (1/st**2)))

    # Calculate cue-integrated distributions
    yt = gaussian_pdf(x, mt, st)
    y1 = gaussian_pdf(x, m1, sd)
    y2 = gaussian_pdf(x, m2, sd)
    yc1 = gaussian_pdf(x, mc1, sc)
    yc2 = gaussian_pdf(x, mc2, sc)
    dprime = (mc2 - mc1) / sc
    C = -.5 * (mc2 + mc1)

    # Plot distributions
    plt.subplot(1, 3, i+1)
    plt.plot(x, y1, c=color_spec, ls='--', lw=1.5, label='Spectral Cue (Low)')
    plt.plot(x, y2, c=color_spec, ls='-', lw=1.5, label='Spectral Cue (High)')
    plt.plot(x, yt, c=color_temp, ls=':', lw=2.25, label='Temporal Cue (Late)')
    plt.plot(x, yc1, label='Integrated Estimate (Low)', ls='--', c='k', lw=2.25)
    plt.plot(x, yc2, label='Integrated Estimate (High)', c='k', lw=2.25)
    plt.xlim(-5, 5)
    plt.xticks(range(-5, 6), ('$-5$', '$-4$', '$-3$', '$-2$', '$-1$', '$0$', '$+1$', '$+2$', '$+3$', '$+4$', '$+5$'), fontsize=16)
    plt.xlabel('Estimated Pitch Change (JNDs)', size=16)
    plt.ylim(-.005, .5)
    plt.yticks(np.arange(0, .51, .1), fontsize=16)
    plt.ylabel('Probability Density', size=16)
    plt.title(titles[i], size=16)
    plt.text(2.55, .38, '$d^\\prime$ = %.02f\n\n$C$ = %.02f' % (dprime, C), size=16, bbox=dict(facecolor='w', edgecolor='black', boxstyle='round,pad=0.7'))
    plt.grid(True)
    
    # Add legend below figure
    if i == 0:
        fig.legend(loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.1), fontsize=16)
        fig.subplots_adjust(bottom=0.2)

fig.tight_layout()
fig.savefig('figures/cue_integration.svg', bbox_inches='tight')
fig.savefig('figures/cue_integration.pdf', bbox_inches='tight')

# Scratchwork

In [ ]:
time = np.linspace(0, 100, 101)  # normalized time from 0 to 1
x = np.linspace(-20, 20, 100)

# Define mean and standard deviation of individual cue estimates over time
ms = 1
ss_arr = np.geomspace(24, 1, len(time))
mt = -5
st_arr = np.geomspace(12, 8, len(time))

# Calculate mean and standard deviation of combined estimate over time
mc_arr = (ms * st_arr**2 + mt * ss_arr**2) / (ss_arr**2 + st_arr**2)
sc_arr = np.sqrt(1 / ((1/ss_arr**2) + (1/st_arr**2)))

# Plot estimates at various points in time
for i, p in enumerate((0, 25, 50, 75, 100)):

    ys = gaussian_pdf(x, ms, ss_arr[p])
    yt = gaussian_pdf(x, mt, st_arr[p])
    yc = gaussian_pdf(x, mc_arr[p], sc_arr[p])

    plt.subplot(1, 5, i + 1)
    plt.plot(x, ys, label='Spectral Cue', ls='--', c='k')
    plt.plot(x, yt, label='Temporal Cue', ls=':', c='k')
    plt.plot(x, yc, label='Integrated Cues', ls='-', c='k')
    plt.title('Time Elapsed = %i%%; MLE = %.03f' % (p, mc_arr[p]))
    plt.xlabel('Estimated Pitch Change (JNDs)')
    plt.ylabel('Probability Density')
    #plt.grid(True)
    plt.legend()
    plt.xlim(-10, 10)
    plt.ylim(-.005, .45)

plt.gcf().set_size_inches(24, 4)
plt.tight_layout()

In [ ]:
plt.subplot(121)
plt.plot(time, ss_arr, ls='--', c='k', label='Spectral Cue')
plt.plot(time, st_arr, ls=':', c='k', label='Temporal Cue')
plt.plot(time, sc_arr, ls='-', c='k', label='Integrated Cues')
plt.xlabel('Time Elapsed (%)')
plt.ylabel('Standard Deviation of Estimate')
plt.xlim(0, 100)
plt.ylim(0, 25)
plt.xticks(range(0, 101, 25))
plt.legend()

plt.subplot(122)
plt.axhline(ms, ls='--', c='k', label='Spectral MLE')
plt.axhline(mt, ls=':', c='k', label='Temporal MLE')
plt.plot(time, mc_arr, ls='-', c='k', label='Integrated MLE')
plt.xlabel('Time Elapsed (%)')
plt.ylabel('Estimated Pitch Change (JNDs)')
plt.xlim(0, 100)
plt.xticks(range(0, 101, 25))
plt.legend()

plt.gcf().set_size_inches(10, 4)
plt.tight_layout()

In [ ]:
time = np.linspace(0, 100, 101)  # normalized time from 0 to 1
x = np.linspace(-20, 20, 100)

# Define mean and standard deviation of individual cue estimates over time
ms = -1
ss_arr = np.geomspace(24, 1, len(time))
mt = -5
st_arr = np.geomspace(12, 8, len(time))

# Calculate mean and standard deviation of combined estimate over time
mc_arr = (ms * st_arr**2 + mt * ss_arr**2) / (ss_arr**2 + st_arr**2)
sc_arr = np.sqrt(1 / ((1/ss_arr**2) + (1/st_arr**2)))

# Plot estimates at various points in time
for i, p in enumerate((0, 25, 50, 75, 100)):

    ys = gaussian_pdf(x, ms, ss_arr[p])
    yt = gaussian_pdf(x, mt, st_arr[p])
    yc = gaussian_pdf(x, mc_arr[p], sc_arr[p])

    plt.subplot(1, 5, i + 1)
    plt.plot(x, ys, label='Spectral Cue', ls='--', c='k')
    plt.plot(x, yt, label='Temporal Cue', ls=':', c='k')
    plt.plot(x, yc, label='Integrated Cues', ls='-', c='k')
    plt.title('Time Elapsed = %i%%; MLE = %.03f' % (p, mc_arr[p]))
    plt.xlabel('Estimated Pitch Change (JNDs)')
    plt.ylabel('Probability Density')
    #plt.grid(True)
    plt.legend()
    plt.xlim(-10, 10)
    plt.ylim(-.005, .45)

plt.gcf().set_size_inches(24, 4)
plt.tight_layout()

In [ ]:
time = np.linspace(0, 100, 101)  # normalized time from 0 to 1
x = np.linspace(-20, 20, 100)

# Define mean and standard deviation of individual cue estimates over time
ms = .5
ss_arr = np.geomspace(24, 1, len(time))
mt = -5
st_arr = np.geomspace(12, 8, len(time))

# Calculate mean and standard deviation of combined estimate over time
mc_arr = (ms * st_arr**2 + mt * ss_arr**2) / (ss_arr**2 + st_arr**2)
sc_arr = np.sqrt(1 / ((1/ss_arr**2) + (1/st_arr**2)))

# Plot estimates at various points in time
for i, p in enumerate((0, 25, 50, 75, 100)):

    ys = gaussian_pdf(x, ms, ss_arr[p])
    yt = gaussian_pdf(x, mt, st_arr[p])
    yc = gaussian_pdf(x, mc_arr[p], sc_arr[p])

    plt.subplot(1, 5, i + 1)
    plt.plot(x, ys, label='Spectral Cue', ls='--', c='k')
    plt.plot(x, yt, label='Temporal Cue', ls=':', c='k')
    plt.plot(x, yc, label='Integrated Cues', ls='-', c='k')
    plt.title('Time Elapsed = %i%%; MLE = %.03f' % (p, mc_arr[p]))
    plt.xlabel('Estimated Pitch Change (JNDs)')
    plt.ylabel('Probability Density')
    #plt.grid(True)
    plt.legend()
    plt.xlim(-10, 10)
    plt.ylim(-.005, .45)

plt.gcf().set_size_inches(24, 4)
plt.tight_layout()

In [ ]:
time = np.linspace(0, 100, 101)  # normalized time from 0 to 1
x = np.linspace(-20, 20, 100)

# Define mean and standard deviation of individual cue estimates over time
ms = -.5
ss_arr = np.geomspace(24, 1, len(time))
mt = -5
st_arr = np.geomspace(12, 8, len(time))

# Calculate mean and standard deviation of combined estimate over time
mc_arr = (ms * st_arr**2 + mt * ss_arr**2) / (ss_arr**2 + st_arr**2)
sc_arr = np.sqrt(1 / ((1/ss_arr**2) + (1/st_arr**2)))

# Plot estimates at various points in time
for i, p in enumerate((0, 25, 50, 75, 100)):

    ys = gaussian_pdf(x, ms, ss_arr[p])
    yt = gaussian_pdf(x, mt, st_arr[p])
    yc = gaussian_pdf(x, mc_arr[p], sc_arr[p])

    plt.subplot(1, 5, i + 1)
    plt.plot(x, ys, label='Spectral Cue', ls='--', c='k')
    plt.plot(x, yt, label='Temporal Cue', ls=':', c='k')
    plt.plot(x, yc, label='Integrated Cues', ls='-', c='k')
    plt.title('Time Elapsed = %i%%; MLE = %.03f' % (p, mc_arr[p]))
    plt.xlabel('Estimated Pitch Change (JNDs)')
    plt.ylabel('Probability Density')
    #plt.grid(True)
    plt.legend()
    plt.xlim(-10, 10)
    plt.ylim(-.005, .45)

plt.gcf().set_size_inches(24, 4)
plt.tight_layout()

In [ ]:
-.5 * (.415-.569), -.5 * (.908-1.062)

In [ ]:
time = np.linspace(0, 100, 101)  # normalized time from 0 to 1
x = np.linspace(-20, 20, 100)

# Define mean and standard deviation of individual cue estimates over time
ms = 1
ss_arr = np.geomspace(48, 2, len(time))
mt = -5
st_arr = np.geomspace(12, 8, len(time))

# Calculate mean and standard deviation of combined estimate over time
mc_arr = (ms * st_arr**2 + mt * ss_arr**2) / (ss_arr**2 + st_arr**2)
sc_arr = np.sqrt(1 / ((1/ss_arr**2) + (1/st_arr**2)))

# Plot estimates at various points in time
for i, p in enumerate((0, 25, 50, 75, 100)):

    ys = gaussian_pdf(x, ms, ss_arr[p])
    yt = gaussian_pdf(x, mt, st_arr[p])
    yc = gaussian_pdf(x, mc_arr[p], sc_arr[p])

    plt.subplot(1, 5, i + 1)
    plt.plot(x, ys, label='Spectral Cue', ls='--', c='k')
    plt.plot(x, yt, label='Temporal Cue', ls=':', c='k')
    plt.plot(x, yc, label='Integrated Cues', ls='-', c='k')
    plt.title('Time Elapsed = %i%%; MLE = %.03f' % (p, mc_arr[p]))
    plt.xlabel('Estimated Pitch Change (JNDs)')
    plt.ylabel('Probability Density')
    #plt.grid(True)
    plt.legend()
    plt.xlim(-10, 10)
    plt.ylim(-.005, .45)

plt.gcf().set_size_inches(24, 4)
plt.tight_layout()

In [ ]:
time = np.linspace(0, 100, 101)  # normalized time from 0 to 1
x = np.linspace(-20, 20, 100)

# Define mean and standard deviation of individual cue estimates over time
ms = -1
ss_arr = np.geomspace(48, 2, len(time))
mt = -5
st_arr = np.geomspace(12, 8, len(time))

# Calculate mean and standard deviation of combined estimate over time
mc_arr = (ms * st_arr**2 + mt * ss_arr**2) / (ss_arr**2 + st_arr**2)
sc_arr = np.sqrt(1 / ((1/ss_arr**2) + (1/st_arr**2)))

# Plot estimates at various points in time
for i, p in enumerate((0, 25, 50, 75, 100)):

    ys = gaussian_pdf(x, ms, ss_arr[p])
    yt = gaussian_pdf(x, mt, st_arr[p])
    yc = gaussian_pdf(x, mc_arr[p], sc_arr[p])

    plt.subplot(1, 5, i + 1)
    plt.plot(x, ys, label='Spectral Cue', ls='--', c='k')
    plt.plot(x, yt, label='Temporal Cue', ls=':', c='k')
    plt.plot(x, yc, label='Integrated Cues', ls='-', c='k')
    plt.title('Time Elapsed = %i%%; MLE = %.03f' % (p, mc_arr[p]))
    plt.xlabel('Estimated Pitch Change (JNDs)')
    plt.ylabel('Probability Density')
    #plt.grid(True)
    plt.legend()
    plt.xlim(-10, 10)
    plt.ylim(-.005, .45)

plt.gcf().set_size_inches(24, 4)
plt.tight_layout()

In [ ]:
-.5 * (.647-1.235), -.5 * (.908-1.062)